# Full-context temporal convolution package for v009

Definitions only. Execute the unchanged `research/sequence_components_v005.ipynb` definitions first. No assignment data, external resources, fitting, filesystem changes, or training is accessed while defining these cells. The caller owns the original combined47 known inputs, training-only preprocessing, grouped split, masked inspiratory L1, seed, batch256, frozen200 rates, and fixed .5/.5 deep-model pairing.

The hypothesis is that a full-breath convolution encoder may learn smooth pressure-curve errors differently from the compact bidirectional LSTM. This is an **encoder and head architecture package comparison**. It does not preserve common weights or claim to isolate convolution alone. Existing baseline artifacts remain untouched.

`V9TCNPressureModel` projects the F standardized numeric features to **96 channels**, then applies **six residual blocks** with dilation **1,2,4,8,16,32**. Every block has **two symmetric kernel3 Conv1d layers**, each followed by per-position LayerNorm over channels, SiLU, and dropout **.15**. Residual addition preserves the sequence shape. There is no post-residual activation, extra branch, normalization across breaths, batch normalization, attention, pooling, recurrent state, or persistent running statistic. A per-position **96-to-head_width-to-1** SiLU head emits scalar standardized pressure. Production head_width stays32; the original V005 toy head_width is supported for synthetic engine checks.

The theoretical receptive field is **253 positions** (radius126), covering all 80 positions from every output location. Symmetric padding preserves exactly 80 outputs and deliberately allows future known inputs within the same offline breath. There is no causal-mask claim. The input projection sees all original numeric features, including R/C. With the required zero joint-embedding dimension, the appended internal joint R/C code is ignored exactly as in the original scalar baseline; it is not a new learned input. IDs and pressure labels are never prediction arguments. Optional preprocessing is inspected only for feature-name validation, never for pressure values or target statistics; scaling remains caller owned and training only.

Public APIs: `v9_tcn_model(spec, n_features, prep=None)` and `v9_tcn_selfcheck(device='cpu')`. The existing canonical V005 spec is retained as recipe metadata; this fixed TCN package uses its `head_width` and requires `dropout=.15`, no R/C-specific heads, and no joint embedding. Baseline `family`, hidden/layer/width/residual/concatenation fields do not change the fixed TCN encoder. At F47 and head32 the package has **342,977 trainable parameters**. Different initialization and dropout consumption prevent any claim of identical baseline trajectories.

`v9_tcn_source_guard(repository_root='.', expected_notebook_sha256=None)` explicitly checks this component's identity/implementation cells, the unchanged V005 engine hash, and optionally the caller-frozen complete-notebook hash. Defining it performs no reads. Source definitions must remain frozen during a stage.

The explicit synthetic selfcheck verifies shape and exact80-length rejection, zero inputs, receptive field and actual far-future influence, numeric R/C use and internal-code irrelevance, absence of ID/pressure inputs, independent breaths/no persistent state, deterministic forward/backward, active .15 dropout, repeatable evaluation, finite gradients, and strict checkpoint reconstruction. It restores caller RNG/backend settings. CPU evidence does not establish CUDA determinism, epoch-boundary resume, batch256 throughput, or the 11GB memory envelope. Run the shared trainer resume proof and paired actual-GPU profiling before approved-data fits; stop for review at the fixed resource gates.


In [ ]:
# Required immutable V005 definitions and component identity. Definitions only.
_V9_TCN_REQUIRED = ('canonical_model_spec', 'seed_sequence', '_sequence_device',
    '_rng_state', '_restore_rng', 'torch', 'nn', 'np', 'hashlib', 'json', 'Path', 'V5_STEPS')
_missing_v9_tcn = [name for name in _V9_TCN_REQUIRED if name not in globals()]
if _missing_v9_tcn:
    raise RuntimeError('Execute unchanged V005 sequence definitions first: ' + ', '.join(_missing_v9_tcn))
V9_TCN_VERSION = 'v009-full-context-tcn96x6-k3-v1'
V9_TCN_NOTEBOOK = 'research/tcn_components_v009.ipynb'
V9_TCN_BASE_NOTEBOOK = 'research/sequence_components_v005.ipynb'
V9_TCN_BASE_SHA256 = 'd20e92eec5889b094a87486f876c3e1a8ae1d7eb2aa096fa5dd530b64c914b4e'
V9_TCN_CHANNELS = 96
V9_TCN_DILATIONS = (1, 2, 4, 8, 16, 32)
V9_TCN_DROPOUT = .15
V9_TCN_RECEPTIVE_FIELD = 1 + 4 * sum(V9_TCN_DILATIONS)
V9_TCN_CODE_SHA256 = 'e1f12ef0353a1adf0e3ccb2570fc8666b2c2be899a6aa2db676f8531c7ad3624'


def v9_tcn_source_guard(repository_root='.', expected_notebook_sha256=None):
    """Explicit source guard; the caller must execute these same verified bytes."""
    repository_root = Path(repository_root)
    source = (repository_root / V9_TCN_NOTEBOOK).read_bytes()
    notebook = json.loads(source)
    identity = notebook.get('metadata', {}).get('tcn_component', {})
    if identity != dict(version=V9_TCN_VERSION, notebook=V9_TCN_NOTEBOOK,
                        implementation_sha256=V9_TCN_CODE_SHA256):
        raise ValueError('V009 TCN notebook identity/hash metadata mismatch')
    cells = [cell for cell in notebook['cells']
        if cell.get('metadata', {}).get('v9_tcn_implementation')]
    if len(cells) != 2:
        raise ValueError('V009 TCN implementation-cell layout changed')
    payload = '\n\n'.join(''.join(cell['source']) for cell in cells)
    code_sha256 = hashlib.sha256(payload.encode()).hexdigest()
    if code_sha256 != V9_TCN_CODE_SHA256:
        raise ValueError('V009 TCN implementation SHA256 mismatch')
    base_sha256 = hashlib.sha256((repository_root / V9_TCN_BASE_NOTEBOOK).read_bytes()).hexdigest()
    if base_sha256 != V9_TCN_BASE_SHA256:
        raise ValueError('V009 TCN requires the unchanged V005 sequence engine')
    notebook_sha256 = hashlib.sha256(source).hexdigest()
    if expected_notebook_sha256 is not None and notebook_sha256 != expected_notebook_sha256:
        raise ValueError('V009 TCN full-notebook SHA256 mismatch')
    return dict(version=V9_TCN_VERSION, notebook=V9_TCN_NOTEBOOK,
        notebook_sha256=notebook_sha256, implementation_sha256=code_sha256,
        base_sha256=base_sha256, external_notebook_hash_checked=expected_notebook_sha256 is not None)


In [ ]:
# Full-context convolution definitions. No labels, IDs or persistent breath state.
def _v9_tcn_feature_names(prep, n_features):
    if prep is None:
        return None
    names = prep.get('feature_names')
    if names is None or len(names) != n_features or len(set(names)) != n_features:
        raise ValueError('TCN preprocessing requires one unique known name per numeric feature')
    names = tuple(names)
    if names[:2] != ('R', 'C'):
        raise ValueError('The original numeric R and C must remain the first two features')
    forbidden = {'id', 'breath_id', 'pressure', 'target', 'label'}
    if any(str(name).lower() in forbidden or 'pressure' in str(name).lower() for name in names):
        raise ValueError('IDs, labels and pressure-derived features are not allowed')
    return names


class _V9TemporalResidualBlock(nn.Module):
    """Two symmetric convolutions, each normalized independently per position."""
    def __init__(self, channels, dilation):
        super().__init__()
        self.dilation = int(dilation)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3,
            dilation=self.dilation, padding=self.dilation)
        self.norm1 = nn.LayerNorm(channels)
        self.activation1 = nn.SiLU()
        self.dropout1 = nn.Dropout(V9_TCN_DROPOUT)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3,
            dilation=self.dilation, padding=self.dilation)
        self.norm2 = nn.LayerNorm(channels)
        self.activation2 = nn.SiLU()
        self.dropout2 = nn.Dropout(V9_TCN_DROPOUT)

    def forward(self, sequence):
        transformed = self.conv1(sequence.transpose(1, 2)).transpose(1, 2)
        transformed = self.dropout1(self.activation1(self.norm1(transformed)))
        transformed = self.conv2(transformed.transpose(1, 2)).transpose(1, 2)
        transformed = self.dropout2(self.activation2(self.norm2(transformed)))
        return sequence + transformed


class V9TCNPressureModel(nn.Module):
    """Fixed six-block96-channel encoder package with an input-only scalar head.

    Input is already transformed by the caller's training-only V005 preprocessor:
    F standardized numeric channels followed by the unused internal joint R/C code.
    All convolutions act within a single complete known input breath. There are no
    running statistics or state shared across calls or across different breaths.
    """
    def __init__(self, model_spec, n_features, prep=None):
        super().__init__()
        self.spec = spec = canonical_model_spec(model_spec)
        self.n_features = int(n_features)
        if self.n_features < 2:
            raise ValueError('At least the original physical numeric R and C are required')
        if spec['rc_heads'] or spec['joint_embedding_dim']:
            raise ValueError('V009 TCN requires a scalar head and no joint R/C embedding')
        if spec['dropout'] != V9_TCN_DROPOUT:
            raise ValueError('V009 TCN fixes all block dropout at .15')
        self.feature_names = _v9_tcn_feature_names(prep, self.n_features)
        self.input_projection = nn.Linear(self.n_features, V9_TCN_CHANNELS)
        self.blocks = nn.ModuleList(_V9TemporalResidualBlock(V9_TCN_CHANNELS, dilation)
            for dilation in V9_TCN_DILATIONS)
        self.head = nn.Sequential(nn.Linear(V9_TCN_CHANNELS, spec['head_width']),
            nn.SiLU(), nn.Linear(spec['head_width'], 1))
        self.receptive_field = V9_TCN_RECEPTIVE_FIELD

    def forward(self, x):
        if x.ndim != 3 or x.shape[0] < 1 or x.shape[1] != V5_STEPS or x.shape[2] != self.n_features + 1:
            raise ValueError('Expected nonempty complete known input breaths [B,80,F+1]')
        # Retain every numeric feature, including R/C; ignore only the internal code.
        sequence = self.input_projection(x[..., :self.n_features])
        for block in self.blocks:
            sequence = block(sequence)
        return self.head(sequence).squeeze(-1)


def v9_tcn_model(model_spec, n_features, prep=None):
    """Three-argument shared-trainer factory; preprocessing is feature-name-only."""
    return V9TCNPressureModel(model_spec, n_features, prep)


In [ ]:
# Optional generated-input proof. Never executed during notebook definition.
def v9_tcn_selfcheck(device='cpu'):
    """Test the actual fixed96x6 package, restoring RNG and backend state.

    This small synthetic test does not replace production batch256 profiling or
    the shared trainer's exact epoch-boundary resume check on the actual GPU.
    """
    import inspect
    import platform

    selected = _sequence_device(device)
    generator = torch.Generator().manual_seed(9009)
    rng_before = _rng_state(generator)
    backend_before = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        cudnn_benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    def rejects(name, action):
        try:
            action()
        except (ValueError, TypeError, RuntimeError):
            checks[name] = True
        else:
            raise AssertionError(name + ' did not reject')

    try:
        seed_sequence(20261102)
        spec = canonical_model_spec(dict(family='lstm', hidden=128, layers=2,
            dropout=.15, bidirectional=True, head_width=32))
        names = ['R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt'] + [
            f'known_synthetic_feature_{i}' for i in range(7, 47)]
        prep = dict(feature_names=names)
        model = v9_tcn_model(spec, 47, prep).to(selected)
        parameter_count = sum(parameter.numel() for parameter in model.parameters())
        check('production_parameter_count_342977', parameter_count == 342977)
        check('fixed_projection47_to96_scalar_head32',
            model.input_projection.in_features == 47 and model.input_projection.out_features == 96
            and model.head[0].in_features == 96 and model.head[0].out_features == 32
            and model.head[-1].out_features == 1)
        convolutions = [module for module in model.modules() if isinstance(module, nn.Conv1d)]
        check('six_blocks_two_symmetric_kernel3_convolutions_each', len(model.blocks) == 6
            and len(convolutions) == 12 and all(module.kernel_size == (3,)
                and module.padding == module.dilation for module in convolutions)
            and [block.dilation for block in model.blocks] == [1, 2, 4, 8, 16, 32])
        norms = [module for module in model.modules() if isinstance(module, nn.LayerNorm)]
        check('twelve_layernorms_per_position_channels_only', len(norms) == 12
            and all(module.normalized_shape == (96,) for module in norms))
        dropouts = [module for module in model.modules() if isinstance(module, nn.Dropout)]
        check('twelve_explicit_dropouts_all_015', len(dropouts) == 12
            and all(module.p == .15 for module in dropouts))
        check('no_batchnorm_or_recurrent_modules', not any(isinstance(module,
            (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.RNN, nn.LSTM, nn.GRU))
            for module in model.modules()))
        check('no_running_statistic_or_state_buffers', not list(model.named_buffers()))
        check('receptive_field253_radius126_covers80', model.receptive_field == 253
            and (model.receptive_field-1)//2 >= V5_STEPS-1)

        x = torch.randn(3, V5_STEPS, 48, device=selected)
        x[..., -1] = torch.tensor([0., 4., 8.], device=selected)[:, None]
        model.eval()
        with torch.no_grad():
            prediction = model(x)
            check('finite_scalar_output_for_exact80_positions', prediction.shape == (3, 80)
                and torch.isfinite(prediction).all())
            check('evaluation_repeat_exact', torch.equal(prediction, model(x)))
            check('zero_known_inputs_finite_exact80_output',
                torch.isfinite(model(torch.zeros_like(x))).all()
                and model(torch.zeros_like(x)).shape == (3, 80))
            code_changed = x.clone(); code_changed[..., -1] += 123.
            check('internal_joint_code_ignored_exactly', torch.equal(prediction, model(code_changed)))
            changed = x.clone(); changed[1, :, :47] += 1.75
            isolated = model(changed)
            check('different_breath_cannot_change_predictions',
                torch.equal(prediction[0], isolated[0]) and torch.equal(prediction[2], isolated[2]))
            check('known_inputs_change_own_prediction', not torch.equal(prediction[1], isolated[1]))
            model(changed)
            check('no_persistent_state_across_calls', torch.equal(prediction, model(x)))

        # A gradient from output position0 to input position79 is direct evidence
        # of actual full offline context, beyond the receptive-field calculation.
        context_input = x.detach().clone().requires_grad_(True)
        context_gradient, = torch.autograd.grad(model(context_input)[0, 0], context_input)
        check('first_output_uses_far_future_known_input',
            context_gradient[0, -1, :47].abs().sum() > 0)
        check('numeric_R_and_C_are_model_inputs',
            context_gradient[0, :, 0].abs().sum() > 0 and context_gradient[0, :, 1].abs().sum() > 0)
        check('internal_joint_code_has_exactly_zero_input_gradient',
            torch.count_nonzero(context_gradient[..., -1]) == 0)
        check('backward_context_cannot_cross_breaths',
            torch.count_nonzero(context_gradient[1:]) == 0)

        model.train()
        dropout_rng = _rng_state(generator)
        with torch.no_grad():
            dropout_a = model(x)
            dropout_b = model(x)
            _restore_rng(dropout_rng, generator)
            dropout_repeated = model(x)
        check('training_dropout_is_active', not torch.equal(dropout_a, dropout_b))
        check('same_rng_dropout_forward_exact', torch.equal(dropout_a, dropout_repeated))
        target = .2*x[..., 2] + .1*torch.sin(x[..., 3]) + .4
        optimizer = torch.optim.Adam(model.parameters(), lr=.0015, weight_decay=0.)
        backward_rng = _rng_state(generator)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(x)
        loss = (prediction-target).abs().mean()
        loss.backward()
        gradients = {name: parameter.grad.detach().clone() for name, parameter in model.named_parameters()}
        check('finite_loss_and_all_parameter_gradients', torch.isfinite(loss)
            and all(torch.isfinite(gradient).all() for gradient in gradients.values()))
        check('all_convolution_weights_receive_nonzero_gradients', all(
            parameter.grad is not None and parameter.grad.abs().sum() > 0
            for name, parameter in model.named_parameters() if '.conv' in name and name.endswith('weight')))
        check('projection_and_scalar_head_receive_nonzero_gradients',
            model.input_projection.weight.grad.abs().sum() > 0 and model.head[0].weight.grad.abs().sum() > 0)
        _restore_rng(backward_rng, generator)
        optimizer.zero_grad(set_to_none=True)
        repeated_loss = (model(x)-target).abs().mean()
        repeated_loss.backward()
        check('same_rng_backward_gradients_exact', torch.equal(loss.detach(), repeated_loss.detach())
            and all(torch.equal(gradients[name], parameter.grad)
                for name, parameter in model.named_parameters()))
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            trained_prediction = model(x)
            check('post_update_eval_repeat_exact', torch.equal(trained_prediction, model(x)))
            # Target-related extra preprocessing entries have no effect on model
            # reconstruction; only the known feature-name schema is inspected.
            poisoned_prep = dict(prep, target_mean=float('nan'), target_scale=-999.,
                pressure=np.array([float('inf')]))
            clone = v9_tcn_model(spec, 47, poisoned_prep).to(selected)
            clone.load_state_dict(model.state_dict(), strict=True); clone.eval()
            check('strict_state_roundtrip_prediction_exact', torch.equal(trained_prediction, clone(x)))
            check('target_statistics_never_read_by_model', torch.equal(trained_prediction, clone(x)))
        toy = v9_tcn_model(dict(spec, hidden=8, layers=2, head_width=8), 7,
            dict(feature_names=names[:7])).to(selected)
        check('tiny_trainer_spec_keeps_fixed_encoder_and_toy_head',
            toy.head[0].out_features == 8 and toy.input_projection.out_features == 96
            and len(toy.blocks) == 6)
        check('forward_contract_known_inputs_only', tuple(
            inspect.signature(V9TCNPressureModel.forward).parameters) == ('self', 'x'))
        check('factory_contract_compatible_prep_argument', tuple(
            inspect.signature(v9_tcn_model).parameters) == ('model_spec', 'n_features', 'prep'))
        rejects('reject_label_forward_argument', lambda: model(x, target))
        rejects('reject_zero_length_sequence', lambda: model(x[:, :0]))
        rejects('reject79_steps', lambda: model(x[:, :79]))
        rejects('reject81_steps', lambda: model(torch.cat((x, x[:, :1]), dim=1)))
        rejects('reject_empty_batch', lambda: model(x[:0]))
        rejects('reject_wrong_feature_width', lambda: model(x[..., :-1]))
        rejects('reject_wrong_rank', lambda: model(x[0]))
        rejects('reject_different_dropout', lambda: v9_tcn_model(dict(spec, dropout=.25), 47, prep))
        rejects('reject_joint_embedding', lambda: v9_tcn_model(dict(spec, joint_embedding_dim=4), 47, prep))
        rejects('reject_rc_specific_heads', lambda: v9_tcn_model(dict(spec, rc_heads=True), 47, prep))
        for forbidden in ('id', 'breath_id', 'pressure', 'pressure_lag1'):
            invalid_names = list(names); invalid_names[7] = forbidden
            rejects('reject_feature_' + forbidden, lambda names=invalid_names:
                v9_tcn_model(spec, 47, dict(feature_names=names)))
        check('strict_determinism_remains_enabled', torch.are_deterministic_algorithms_enabled()
            and not torch.is_deterministic_algorithms_warn_only_enabled())
        return dict(schema='v9_tcn_selfcheck', version=V9_TCN_VERSION,
            implementation_sha256=V9_TCN_CODE_SHA256,
            data_scope='generated_synthetic_known_inputs_only; no assignment-data reads or fits',
            python_version=platform.python_version(), torch_version=str(torch.__version__),
            device=str(selected), cuda_version=torch.version.cuda,
            cuda_name=torch.cuda.get_device_name(selected) if selected.type == 'cuda' else None,
            model_spec=spec, n_features=47, synthetic_shape=[3, 80, 48],
            parameter_count=int(parameter_count), channels=96, dilations=list(V9_TCN_DILATIONS),
            receptive_field=V9_TCN_RECEPTIVE_FIELD, common_baseline_initialization=False,
            far_future_input_gradient_l1=float(context_gradient[0, -1, :47].abs().sum()),
            checks=checks, check_count=len(checks), passed=all(checks.values()),
            gpu_profile_or_engine_resume_proof=False)
    finally:
        _restore_rng(rng_before, generator)
        torch.set_num_threads(backend_before['threads'])
        torch.use_deterministic_algorithms(backend_before['deterministic'],
            warn_only=backend_before['warn_only'])
        torch.backends.cudnn.benchmark = backend_before['cudnn_benchmark']
        torch.backends.cudnn.deterministic = backend_before['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend_before['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend_before['cudnn_tf32']
